### This EXTRACT_CORPUS notebook 

- Extracts the works for each journal into the cache

- Load the works into the database after  
    - Filters works into a flat table (work_id, doi, source, host, citation_count etc)  
    - Flattens the authorships table for each work (author_id, institution_id etc)  
    - Filters the reference list to make the cited table (When inverted these are the endogenous citations)  

Note that at this stage the works have been filtered by publication date and type (articles, etc)


In [31]:
%run common_setup.ipynb

In [32]:
class ArticlesETL(SetUp):

    def __init__(self):
        super().__init__()
        return

    # def _extract_works_by_journal(self):
    #     # Extract OA works for the journal set, for publication years 2010+ to now
    #     self._match_journals()
    #     hold = []
    #     for row in self.journals.itertuples():
    #         if row.Index > 64:
    #             continue
    #         source_id = row.source_id
    #         if source_id is None:
    #             print(f'source_is IS None FOR {row= }')
    #             continue
    #         reader = rf'Works().filter(primary_location={{"source": {{"id": "{source_id}"}}}}).filter(publication_year=">2009")'
    #         if isinstance(oa := self._cache_manager(task=reader), pd.DataFrame) and len(oa) > 0:
    #             oa = self._filter_works(works=oa)
    #             self._sql_appender(df=oa, row=row.Index)
    #             print(f'APPENDED {row.Index = } {oa.shape = }')
    #         else:
    #             print(f'OpenAlex does not have articles for {source_id = } {row.display_name = }')
    #         if row.Index % 1 == 0:
    #             print(f'{row.Index}/{len(self.journals)} completed')             
    #     return

    def extract_works_by_journal(self):

        sql = """
        -- ETL TO EXTRACT works FOR journal list
        -- ======================================
            CREATE OR REPLACE TABLE project.raw AS
            SELECT DISTINCT ON (doi)
                    w.*,
                    lower(title) AS lower_title,
                    IF (first_page = last_page, 0, try_cast(last_page AS INT) - try_cast(first_page AS INT)) AS page_count
            FROM project.jcr_matches
            LEFT JOIN works.works w
            USING (source_id)
            WHERE (first_page IS NULL OR page_count > 1)
                    AND doi IS NOT NULL
                    AND publication_year > 2009
                    AND contains(lower_title, 'editor') = false 
                    AND contains(lower_title, 'issue information') = false
                    AND contains(lower_title, 'index') = false
                    AND contains(lower_title, 'book review') = false
                    AND contains(lower_title, 'isbn') = false
                    AND contains(lower_title, 'calendar of events') = false
                    AND contains(lower_title, 'notes on contributors') = false
                    AND regexp_matches(lower_title, '^announcements$') = false
                    AND regexp_matches(lower_title, '^acknowledgement') = false
                    AND regexp_matches(lower_title, '^vol[.u ] ') = false
                    AND regexp_matches(lower_title, '^focus on authors$') = false
                    AND regexp_matches(lower_title, '^publications received$') = false
                    AND list_contains(['article', 'review', 'letter'], type) = true
                    AND work_id != 'https://openalex.org/works/W4285719527'
            -- ORDER BY page_count
        """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) FROM project.raw").show()
        self.db.sql("SELECT * FROM project.raw").show()
        return


    # def _match_journals(self):
    #     sql = """
    #         -- ETL TO MATCH journal list to OA source_id VIA ISSN
    #         -- ==================================================
    #         SELECT DISTINCT source_id, 
    #                 journal,
    #                 issn_l,
    #                 pub,
    #                 count(issn_l) AS duplicated
    #         FROM project.jcr_matches
    #         GROUP BY ALL
    #         ORDER BY duplicated DESC, pub DESC
    #     """
    #     self.journals = self.db.sql(sql).df().reset_index(drop=True)
    #     print(f'{self.journals.shape = }\n{self.journals.head()}')
    #     return

    # def _filter_works(self, works=None):
    #     keep_columns = ["id", "doi", "title", "publication_year", "primary_location", "type",
    #             "countries_distinct_count", "institutions_distinct_count", "fwci", "has_fulltext", 
    #             "cited_by_count", "biblio", "is_retracted", "is_paratext", 
    #             "referenced_works_count", "cited_by_api_url", "updated_date", "created_date", 
    #             "authorships", "referenced_works", "topics"]
    #     cols = [c for c in works.columns if c in keep_columns] + [c for c in works.columns if 'biblio.' in c or 'primary_location.source' in c or 'primary_topic.' in c]
    #     works = works[cols] #.fillna(' ')
    #     condition1 = works['is_paratext'] == False
    #     condition2 = works['is_retracted'] == False
    #     condition3 = works['referenced_works_count'] != 0
    #     condition4 = np.array([t in {'article', 'review', 'letter'} for t in works['type']])
    #     condition5 = len(works['authorships']) > 0
    #     works = works.loc[condition1 & condition2 & condition3 & condition4 & condition5].copy()
    #     cast_to_string = ['biblio.volume', 'biblio.issue', 'biblio.first_page', 'biblio.last_page']
    #     works[cast_to_string] = works[cast_to_string].astype(str)
    #     return works[works.id != 'https://openalex.org/works/W4285719527'] # THIS IS A DISCARDED WORK WITH ENORMOUS CITATIONS
    
    # def _sql_appender(self, df=None, row=None):
    #     self._drop_older_duplicated_rows(df=df)
    #     try:
    #         if row == 0:
    #             sql = "CREATE OR REPLACE TABLE project.raw AS (SELECT * FROM df)"
    #         else:
    #             sql = "INSERT INTO project.raw BY NAME (SELECT * FROM df)"
    #         self.db.sql(sql)
    #     except Exception as e:
    #         print(f'CONVERTING df to SQL table project.raw {row = } {e = }')           
    #         print(f'{df.shape = }\n{df.columns = }\n{df.head()}')
    #     return
    
    # def _drop_older_duplicated_rows(self, df=None):
    #     rows = df.shape[0]
    #     df = df.sort_values("updated_date", ascending=False).drop_duplicates(subset='id')
    #     if rows != df.shape[0]:
    #         print(f'DROP OLDER ROWS {df.shape = } due to duplicated update_date')
    #     return
 
    def duplicate_db_as_backup(self):
        self.db.sql("BEGIN TRANSACTION; COPY FROM DATABASE project TO backup; COMMIT;")
        return

In [33]:
class ExtractAuthorshipsReferencesTopics(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def authorships_etl(self):
        print("authorships")
        sql = """
            CREATE OR REPLACE TABLE project.authorships AS  
                (SELECT work_id,
                        author_id,
                        unnest(authorship.institutions).id AS institution_id,
                        unnest(authorship.institutions).country_code AS country_code,    
                    FROM
                    (SELECT id AS work_id,
                            unnest(authorships).author.id AS author_id,
                            unnest(authorships) AS authorship,
                        FROM project.raw
                    )
                )
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(DISTINCT work_id), count(DISTINCT author_id), count(DISTINCT institution_id) FROM project.authorships").show()
        return
    
    def references_etl(self):
        print("references")
        sql = """
            CREATE OR REPLACE TABLE project.citer_cited AS
            WITH
            citer_cited_CTE AS
                (SELECT id AS citer_id,
                        publication_year AS citer_year,
                        unnest(referenced_works) AS cited_id
                    FROM project.raw
                ),
            citer_cited_filtered_CTE AS
                (SELECT DISTINCT citer_id,
                        citer_year,
                        cited_id,
                        publication_year AS cited_year
                    FROM citer_cited_CTE
                    RIGHT JOIN project.raw
                    ON cited_id = id
                    WHERE cited_id NOT NULL
                )

            SELECT *,
                    cited_year - citer_year - 1 AS delta_t
                FROM citer_cited_filtered_CTE
                WHERE delta_t <= 0
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(DISTINCT citer_id), count(DISTINCT cited_id) FROM project.citer_cited").show()
        return
    
    def topics_etl(self):
        print("topics")
        sql = """
            CREATE OR REPLACE TABLE project.topics AS
                SELECT id AS work_id,
                        "primary_topic.id" AS topic_id,
                        "primary_topic.display_name" AS topic_name,
                        "primary_topic.score" AS topic_score,
                        "primary_topic.domain".id AS domain_id,
                        "primary_topic.field".id AS field_id,
                        "primary_topic.subfield".id AS subfield_id
                FROM project.raw
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) FROM project.topics").show()
        return

In [34]:
class  ExtractAuthors(SetUp):

    def __init__(self):
        super().__init__()
        return

    def _extract_author_ids(self):
        df = self.db.sql("SELECT DISTINCT author_id FROM project.authorships").df()
        self.author_ids = [i.replace('https://openalex.org/', '') for i in sorted(df.author_id)] #[:256]
        return

    def extract_authors(self):
        # Extract OA authors for the journal set
        self._extract_author_ids()
        hold = []
        block_length = 100
        start = 0
        block_total = len(self.author_ids)//block_length + 1
        print(f'extract authors {start = } {block_length = } {block_total = }')
        for block_count in range(block_total):
            authors = '|'.join(self.author_ids[start: start+block_length])
            start = start + block_length
            reader = rf'Authors().filter(id="{authors}")'
            if isinstance(oa := self._cache_manager(task=reader), pd.DataFrame) and len(oa) > 0:
                # print(f'EXTRACTED {len(oa) = } authors FOR {authors = }')
                # print(f'{oa.shape = }\n{oa.head()}')
                hold.append(oa)
            else:
                print(f'OpenAlex does not have authors for {block_count=  } {block_count*block_length = } {authors = } {oa.shape = }')
            if block_count % 50 == 0:
                print(f'{block_count = } {block_count*block_length = } {block_count*block_length}/{len(self.author_ids)} completed')
        self._load_authors(hold=hold)
        return
    
    def _load_authors(self, hold=None):
        df = pd.concat(hold, axis=0).rename(columns={'id': 'author_id', 'display_name': 'author_name'})
        df.columns = [c.replace('summary_stats.', '') for c in df.columns]
        self.db.sql("CREATE OR REPLACE TABLE project.authors AS SELECT * FROM df")
        cols = ['author_id', 'orcid', 'author_name', 'display_name_alternatives', 'works_count', 'cited_by_count', '2yr_mean_citedness', 'h_index', 'i10_index', 'topics']
        df = df[cols]
        df[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in df.author_name]
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE project.authors_full AS SELECT * FROM df")
        self.db.sql("SELECT * FROM project.authors").show()
        self.db.sql("SELECT * FROM project.authors_full").show()
        self.db.sql("SELECT count(*) FROM project.authors_full").show()
        return
    
    def duplicate_db_as_backup(self):
        self.db.sql("BEGIN TRANSACTION; COPY FROM DATABASE project TO backup; COMMIT;")
        return         

In [35]:
def main():

    jetl = ArticlesETL()
    # jetl.extract_works_by_journal()
    print(jetl.db.sql("DESCRIBE TABLE project.raw").df())
    jetl.db.sql("SELECT * FROM project.raw").show()
    sql = """SELECT count(DISTINCT source_id) FROM project.raw"""
    jetl.db.sql(sql).show()
    # # jetl.duplicate_db_as_backup()
    jetl.db.close()

    # ea = ExtractAuthorshipsReferencesTopics()
    # ea.authorships_etl()
    # ea.references_etl()
    # ea.topics_etl()
    # ea.db.close()

    # eauthors = ExtractAuthors()
    # eauthors.extract_authors()
    # eauthors.duplicate_db_as_backup()
    # eauthors.db.close()
    
    # mdas = MatchDomingoAuthorSample()
    # mdas.extract_sample()
    # mdas.match_sample()
    # mdas.load_sample()
    # mdas.db.close()

    # mdss = MatchDomingoSourceSample()
    # mdss.special_issn()
    # mdss.extract_jcr()
    # mdss.extract_journals()
    # mdss.match_sources()
    # mdss.db.close()

In [36]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────┬───────────┐
│   database   │ schema  │         name         │     column_names     │           column_types            │ temporary │
│   varchar    │ varchar │       varchar        │      varchar[]       │             varchar[]             │  boolean  │
├──────────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────┼───────────┤
│ authors      │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ institutions │ main    │ institutions         │ [id, ror, display_…  │ [VARCHAR, VARCHAR, VARCHAR, VAR…  │ false     │
│ institutions │ main    │ ror                  │ [name, institution…  │ [VARCHAR, VARCHAR]                │ false     │
│ project      │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ project      │ main    │ autho